# Laboratorio 1 - Análisis Estadístico Descriptivo e Inferencial
**Dataset**: Diabetic Retinopathy Debrecen  
**Integrantes**: Gonzalo Ahumada, Daniel Muñoz  
**Curso**: Inteligencia Computacional - USACH 2026

## 1. Carga del Dataset

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Descargar el dataset directamente desde UCI Machine Learning Repository
# DOI: https://doi.org/10.24432/C5XP4P
repo = fetch_ucirepo(id=329)

# Unir atributos con la variable objetivo en un único DataFrame
df = pd.concat([repo.data.features, repo.data.targets], axis=1)

# Renombrar columnas con nombres descriptivos
df.columns = [
    'quality', 'pre_screening',
    'ma1', 'ma2', 'ma3', 'ma4', 'ma5', 'ma6',
    'exudate1', 'exudate2', 'exudate3', 'exudate4',
    'exudate5', 'exudate6', 'exudate7', 'exudate8',
    'macula_opticdisc_distance', 'opticdisc_diameter',
    'am_fm_classification', 'Class'
]

print(f'Instancias: {df.shape[0]}, Variables: {df.shape[1]}')
df.head()

## 2. Estadística Descriptiva

Calculamos las métricas básicas para las 19 variables.

In [ ]:
# Separar la variable objetivo del resto de los atributos
X = df.drop(columns=['Class'])
y = df['Class']

# Calcular estadísticos: conteo, media, desviación estándar, mínimo y máximo
tabla_descriptiva = X.agg(['count', 'mean', 'std', 'min', 'max']).T
tabla_descriptiva.columns = ['count', 'mean', 'std', 'min', 'max']

print('Tabla de estadísticos descriptivos:')
print(tabla_descriptiva.round(4).to_string())

## 3. Asimetría y Curtosis de Microaneurismas (ma1 - ma6)

Los microaneurismas son la variable de mayor importancia diagnóstica según la literatura.
Analizamos la forma de su distribución.

In [ ]:
# Seleccionar solo las variables de microaneurismas
ma_vars = ['ma1', 'ma2', 'ma3', 'ma4', 'ma5', 'ma6']

# Calcular asimetría (skewness) y curtosis para cada variable
tabla_forma = pd.DataFrame({
    'Asimetría (skew)': df[ma_vars].skew(),
    'Curtosis':         df[ma_vars].kurt()
})

print('Asimetría y curtosis de microaneurismas:')
print(tabla_forma.round(6).to_string())

## 4. Prueba de Normalidad (Shapiro-Wilk)

Hipótesis nula (H0): los datos siguen una distribución normal.  
Si p-valor < 0.05, se rechaza H0 → distribución NO normal.

In [ ]:
resultados_normalidad = []

for var in ma_vars:
    W, p = stats.shapiro(df[var])
    resultados_normalidad.append({
        'Variable':     var,
        'W':            round(W, 6),
        'p-value':      p,
        'Distribución': 'No Normal' if p < 0.05 else 'Normal'
    })

tabla_normalidad = pd.DataFrame(resultados_normalidad).set_index('Variable')
print('Resultados del test de Shapiro-Wilk:')
print(tabla_normalidad.to_string())

## 5. Detección de Outliers (Método de Tukey)

Se identifican valores fuera del rango [Q1 - 1.5*IQR, Q3 + 1.5*IQR].  
Se verifica la clase diagnóstica de cada outlier detectado.

In [ ]:
def detectar_outliers_tukey(serie):
    """Retorna los índices de los outliers usando el método de Tukey."""
    Q1  = serie.quantile(0.25)
    Q3  = serie.quantile(0.75)
    IQR = Q3 - Q1
    return serie[(serie < Q1 - 1.5 * IQR) | (serie > Q3 + 1.5 * IQR)].index.tolist()

print('Outliers por variable de microaneurismas:\n')
for var in ma_vars:
    indices = detectar_outliers_tukey(df[var])
    clases  = df.loc[indices, 'Class'].tolist() if indices else []
    print(f'{var}: {len(indices)} outlier(s) | índices: {indices}')
    print(f'       Clase de cada outlier: {clases}\n')

## 6. Visualización: Boxplot por Clase (Clase 0 vs Clase 1)

In [ ]:
# Convertir a formato largo para facilitar el gráfico
df_ma = df[ma_vars + ['Class']].melt(id_vars='Class', var_name='Variable', value_name='Conteo')

fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(
    data=df_ma, x='Variable', y='Conteo',
    hue='Class', palette={0: '#2ecc71', 1: '#e67e22'},
    ax=ax
)
ax.set_title('Distribución de microaneurismas por clase diagnóstica')
ax.set_xlabel('Variable')
ax.set_ylabel('Conteo de lesiones')
ax.legend(title='Clase', labels=['0 - Sano', '1 - Retinopatía'])
plt.tight_layout()
plt.savefig('../informe/images/1-boxplot.png', dpi=150)
plt.show()

## 7. Mapa de Calor de Correlaciones (Heatmap)

In [ ]:
# Calcular la matriz de correlación de Pearson incluyendo la variable Class
corr = df.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, linewidths=0.4, ax=ax
)
ax.set_title('Matriz de correlación de Pearson')
plt.tight_layout()
plt.savefig('../informe/images/1-heatmap.png', dpi=150)
plt.show()

## 8. Comparación de Grupos: Prueba Mann-Whitney U

Como los datos NO son normales (confirmado en sección 4), usamos una prueba no paramétrica.  
H0: no hay diferencia entre Clase 0 y Clase 1.  
Si p-valor < 0.05, se rechaza H0 → diferencia estadísticamente significativa.

In [ ]:
grupo_sano = df[df['Class'] == 0]
grupo_rd   = df[df['Class'] == 1]

resultados_mw = []

for var in ma_vars:
    U, p = stats.mannwhitneyu(
        grupo_sano[var], grupo_rd[var], alternative='two-sided'
    )
    resultados_mw.append({
        'Variable':      var,
        'U_statistic':   U,
        'p_value':       p,
        'Significativo': 'Sí' if p < 0.05 else 'No'
    })

tabla_mw = pd.DataFrame(resultados_mw).set_index('Variable')
print('Resultados de la prueba Mann-Whitney U:')
print(tabla_mw.to_string())

## 9. Prueba de Homogeneidad de Varianzas (Test de Levene)

Evalúa si las varianzas de los microaneurismas son iguales entre Clase 0 y Clase 1.  
Se usa el test de Levene porque los datos no siguen una distribución normal.  
H0: las varianzas de ambos grupos son iguales (homocedasticidad).  
Si p-valor < 0.05, se rechaza H0 → varianzas significativamente distintas.

In [ ]:
resultados_levene = []

for var in ma_vars:
    stat, p = stats.levene(grupo_sano[var], grupo_rd[var])
    resultados_levene.append({
        'Variable':        var,
        'Estadístico':     round(stat, 4),
        'p_value':         p,
        'Homocedasticidad': 'Sí (varianzas iguales)' if p >= 0.05 else 'No (varianzas distintas)'
    })

tabla_levene = pd.DataFrame(resultados_levene).set_index('Variable')
print('Resultados del Test de Levene:')
print(tabla_levene.to_string())

## 10. Regresión Logística

Modelo para predecir la presencia de retinopatía diabética.  
Se usan 4 variables para evitar multicolinealidad: ma1, exudate1, macula_opticdisc_distance, opticdisc_diameter.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve, ConfusionMatrixDisplay

# Variables seleccionadas para el modelo
features = ['ma1', 'exudate1', 'macula_opticdisc_distance', 'opticdisc_diameter']

X_model = df[features]
y_model = df['Class']

# División 80% entrenamiento / 20% prueba (semilla=42 para reproducibilidad)
X_train, X_test, y_train, y_test = train_test_split(
    X_model, y_model, test_size=0.2, random_state=42
)

# Entrenar modelo
modelo = LogisticRegression(max_iter=1000, random_state=42)
modelo.fit(X_train, y_train)

# Predicciones
y_pred      = modelo.predict(X_test)
y_pred_prob = modelo.predict_proba(X_test)[:, 1]

# Métricas
auc = roc_auc_score(y_test, y_pred_prob)
cm  = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f'AUC-ROC: {auc:.4f}')
print(f'\nMatriz de confusión:')
print(cm)
print(f'\nVerdaderos Positivos (TP): {tp}')
print(f'Verdaderos Negativos (TN): {tn}')
print(f'Falsos Positivos  (FP): {fp}')
print(f'Falsos Negativos  (FN): {fn}')
print(f'Sensibilidad: {tp/(tp+fn):.1%}')
print(f'Especificidad: {tn/(tn+fp):.1%}')

## 11. Gráficos: Matriz de Confusión y Curva ROC

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Matriz de confusión
ConfusionMatrixDisplay(cm, display_labels=['Sano (0)', 'RD (1)']).plot(ax=ax1, colorbar=False)
ax1.set_title('Matriz de Confusión')

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
ax2.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {auc:.2f}')
ax2.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--', label='Azar')
ax2.set_xlabel('Tasa Falsos Positivos')
ax2.set_ylabel('Tasa Verdaderos Positivos')
ax2.set_title('Curva ROC')
ax2.legend(loc='lower right')

plt.tight_layout()
plt.savefig('../informe/images/1-matrix-rocauc.png', dpi=150)
plt.show()

## 12. Histograma de Residuos del Modelo

Los residuos se calculan como la diferencia entre la clase real y la probabilidad predicha.  
Un histograma centrado en cero indica que el modelo no tiene sesgo sistemático.

In [ ]:
from scipy.stats import gaussian_kde

# Residuos: valor real menos probabilidad predicha
residuos = y_test.values - y_pred_prob

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(residuos, bins=30, density=True, color='steelblue', edgecolor='white', alpha=0.85)

# Curva de densidad suavizada
kde = gaussian_kde(residuos)
x_range = np.linspace(residuos.min(), residuos.max(), 300)
ax.plot(x_range, kde(x_range), color='navy', lw=2)

ax.set_title('Histograma de Residuos - Validación de Normalidad del Modelo')
ax.set_xlabel('Residuos')
ax.set_ylabel('Frecuencia')
ax.axvline(0, color='red', linestyle='--', lw=1, label='Cero')
ax.legend()
plt.tight_layout()
plt.savefig('../informe/images/1-histograms.png', dpi=150)
plt.show()